# Working with the Full Dataset

In the previous notebooks, we worked with a small sample of text. This notebook explores the **full SEC filings dataset** that was restored to your Neo4j Aura instance in Lab 1.

The pre-built knowledge graph contains:
- SEC 10-K filings from multiple companies (Apple, Microsoft, Nvidia, etc.)
- Chunks with embeddings for semantic search
- Extracted entities (Companies, Products, Services, Executives, Risk Factors, etc.)
- Relationships between entities

---

## Prerequisites

Make sure you completed **Lab 1** and restored the backup file to your Neo4j Aura instance. If you haven't, go back to Lab 1 and follow the restore instructions.

## Setup

Import required modules and connect to Neo4j.

In [ ]:
import sys
sys.path.insert(0, '..')

from neo4j import GraphDatabase

from config import get_neo4j_driver, get_embedder

In [ ]:
driver = get_neo4j_driver()
driver.verify_connectivity()
print("Connected to Neo4j successfully!")

## Explore the Full Graph

Let's see what's in the restored database.

### Data Model

The knowledge graph uses this schema:

```
(:Document {path, name})
    ↑
    [:FROM_DOCUMENT]
    |
(:Chunk {text, embedding, index})
    ↑
    [:FROM_CHUNK]
    |
(:Company {name, ticker})
    |
    [:FACES_RISK]→(:RiskFactor {name})
    [:MENTIONS]→(:Product {name})
    [:HAS_METRIC]→(:FinancialMetric {name})
    [:OWNS]←(:AssetManager {name})
```

In [ ]:
def show_graph_summary(driver):
    """Show a summary of the complete graph."""
    with driver.session() as session:
        # Count all node types
        result = session.run("""
            MATCH (n)
            UNWIND labels(n) as label
            WITH label
            RETURN label, count(*) as count
            ORDER BY count DESC
        """)
        print("=== Node Counts ===")
        for record in result:
            print(f"  {record['label']}: {record['count']}")
        
        # Count relationship types
        result = session.run("""
            MATCH ()-[r]->()
            WITH type(r) as type
            RETURN type, count(*) as count
            ORDER BY count DESC
        """)
        print("\n=== Relationship Counts ===")
        for record in result:
            print(f"  {record['type']}: {record['count']}")

show_graph_summary(driver)

## Vector Search with Full Dataset

Now let's run the same queries from the previous notebooks. With more data, we'll get more relevant and diverse results.

In [ ]:
embedder = get_embedder()
print(f"Embedder model: {embedder.model_id}")
print(f"Embedding dimensions: {embedder.dimensions}")

In [ ]:
INDEX_NAME = "chunkEmbeddings"

def vector_search(driver, embedder, query: str, top_k: int = 3):
    """Search for chunks similar to the query."""
    query_embedding = embedder.embed_query(query)
    
    with driver.session() as session:
        result = session.run("""
            CALL db.index.vector.queryNodes($index_name, $top_k, $embedding)
            YIELD node, score
            RETURN node.text as text, node.index as idx, score
            ORDER BY score DESC
        """, index_name=INDEX_NAME, top_k=top_k, embedding=query_embedding)
        
        return list(result)

In [ ]:
queries = [
    "What products does Apple make?",
    "Tell me about iPhone and Mac computers",
    "What services does the company offer?",
    "What are the main risk factors?"
]

for query in queries:
    print(f"\nQuery: \"{query}\"")
    print("-" * 50)
    results = vector_search(driver, embedder, query, top_k=2)
    for i, record in enumerate(results):
        text_preview = record['text'][:150] + "..." if len(record['text']) > 150 else record['text']
        print(f"\n[{i+1}] Score: {record['score']:.4f}")
        print(f"    {text_preview}")

## Explore Entities

With the full dataset, we have many more extracted entities to work with.

In [ ]:
def show_entities(driver):
    """Display extracted entities by type."""
    with driver.session() as session:
        # Get entity counts by label (excluding internal labels)
        result = session.run("""
            MATCH (n)
            WHERE NOT n:Chunk AND NOT n:Document AND NOT n:__KGBuilder__
            WITH labels(n) as lbls
            UNWIND lbls as label
            WITH label
            WHERE NOT label STARTS WITH '__'
            RETURN label, count(*) as count
            ORDER BY count DESC
            LIMIT 10
        """)
        print("=== Entity Counts (Top 10) ===")
        for record in result:
            print(f"  {record['label']}: {record['count']}")

show_entities(driver)

In [ ]:
def list_companies(driver):
    """List all Company entities."""
    with driver.session() as session:
        result = session.run("""
            MATCH (c:Company)
            WHERE c.name IS NOT NULL
            RETURN c.name as name
            ORDER BY c.name
            LIMIT 20
        """)
        print("=== Companies ===")
        for record in result:
            print(f"  - {record['name']}")

list_companies(driver)

## Find Chunks for Entities

With the full dataset, we can find multiple chunks that mention specific entities across different documents.

In [ ]:
def find_chunks_for_entity(driver, entity_name: str, limit: int = 3):
    """Find chunks that mention a specific entity."""
    with driver.session() as session:
        result = session.run("""
            MATCH (e)-[:FROM_CHUNK]->(c:Chunk)
            WHERE toUpper(e.name) CONTAINS toUpper($name)
            RETURN e.name as entity, labels(e)[0] as type, c.text as chunk_text
            LIMIT $limit
        """, name=entity_name, limit=limit)
        
        records = list(result)
        if records:
            print(f"=== Chunks mentioning '{entity_name}' ===")
            for i, record in enumerate(records):
                print(f"\n[{i+1}] Entity: {record['entity']} ({record['type']})")
                text_preview = record['chunk_text'][:200] + "..." if len(record['chunk_text']) > 200 else record['chunk_text']
                print(f"    Chunk: {text_preview}")
        else:
            print(f"No chunks found mentioning '{entity_name}'")

# Find chunks mentioning specific entities
find_chunks_for_entity(driver, "iPhone")

In [ ]:
# Try finding chunks for other entities
find_chunks_for_entity(driver, "Microsoft")

In [ ]:
find_chunks_for_entity(driver, "GPU")

## Explore Relationships

With multiple companies and products, we can see the rich relationship network.

In [ ]:
def show_company_products(driver, company_name: str):
    """Show products mentioned by a company."""
    with driver.session() as session:
        result = session.run("""
            MATCH (c:Company)-[:MENTIONS]->(p:Product)
            WHERE toUpper(c.name) CONTAINS toUpper($name)
              AND p.name IS NOT NULL
            RETURN c.name as company, p.name as product
            ORDER BY p.name
            LIMIT 20
        """, name=company_name)
        
        records = list(result)
        if records:
            print(f"=== Products from '{company_name}' ===")
            for record in records:
                print(f"  {record['company']} -> {record['product']}")
        else:
            print(f"No products found for '{company_name}'")

show_company_products(driver, "Apple")

In [ ]:
show_company_products(driver, "Microsoft")

## Explore Risk Factors

SEC 10-K filings include detailed risk factors. Let's explore what risks companies face.

In [ ]:
def show_company_risks(driver, company_name: str, limit: int = 10):
    """Show risk factors for a company."""
    with driver.session() as session:
        result = session.run("""
            MATCH (c:Company)-[:FACES_RISK]->(r:RiskFactor)
            WHERE toUpper(c.name) CONTAINS toUpper($name)
              AND r.name IS NOT NULL
            RETURN c.name as company, r.name as risk
            ORDER BY r.name
            LIMIT $limit
        """, name=company_name, limit=limit)
        
        records = list(result)
        if records:
            print(f"=== Risk Factors for '{company_name}' ===")
            for record in records:
                print(f"  - {record['risk']}")
        else:
            print(f"No risk factors found for '{company_name}'")

show_company_risks(driver, "Apple")

In [ ]:
# Compare risk factors across companies
def find_shared_risks(driver, company1: str, company2: str):
    """Find risk factors shared by two companies."""
    with driver.session() as session:
        result = session.run("""
            MATCH (c1:Company)-[:FACES_RISK]->(r:RiskFactor)<-[:FACES_RISK]-(c2:Company)
            WHERE toUpper(c1.name) CONTAINS toUpper($name1)
              AND toUpper(c2.name) CONTAINS toUpper($name2)
              AND c1 <> c2
            RETURN r.name as shared_risk
            ORDER BY r.name
            LIMIT 10
        """, name1=company1, name2=company2)
        
        records = list(result)
        if records:
            print(f"=== Risks shared by '{company1}' and '{company2}' ===")
            for record in records:
                print(f"  - {record['shared_risk']}")
        else:
            print(f"No shared risks found between '{company1}' and '{company2}'")

find_shared_risks(driver, "Apple", "Microsoft")

## Asset Manager Ownership

The dataset includes ownership information from SEC 13F filings.

In [ ]:
def show_company_owners(driver, company_name: str):
    """Show asset managers that own a company."""
    with driver.session() as session:
        result = session.run("""
            MATCH (am:AssetManager)-[:OWNS]->(c:Company)
            WHERE toUpper(c.name) CONTAINS toUpper($name)
            RETURN am.managerName as manager, c.name as company
            ORDER BY am.managerName
            LIMIT 15
        """, name=company_name)
        
        records = list(result)
        if records:
            print(f"=== Asset Managers owning '{company_name}' ===")
            for record in records:
                print(f"  - {record['manager']}")
        else:
            print(f"No ownership data found for '{company_name}'")

show_company_owners(driver, "Apple")

In [ ]:
# Find asset managers that own both Apple and Microsoft
def find_common_owners(driver, company1: str, company2: str):
    """Find asset managers that own both companies."""
    with driver.session() as session:
        result = session.run("""
            MATCH (am:AssetManager)-[:OWNS]->(c1:Company),
                  (am)-[:OWNS]->(c2:Company)
            WHERE toUpper(c1.name) CONTAINS toUpper($name1)
              AND toUpper(c2.name) CONTAINS toUpper($name2)
              AND c1 <> c2
            RETURN DISTINCT am.managerName as manager
            ORDER BY manager
            LIMIT 10
        """, name1=company1, name2=company2)
        
        records = list(result)
        if records:
            print(f"=== Asset Managers owning both '{company1}' and '{company2}' ===")
            for record in records:
                print(f"  - {record['manager']}")
        else:
            print(f"No common owners found for '{company1}' and '{company2}'")

find_common_owners(driver, "Apple", "Microsoft")

## Summary

With the full dataset loaded, you can see:

1. **More diverse search results** - Queries now return relevant chunks from multiple companies and documents
2. **Richer entity network** - Many more extracted companies, products, services, and other entities
3. **Cross-document relationships** - Entities are linked across different SEC filings
4. **Better answers** - More context means more accurate and comprehensive responses

This full dataset is what you'll use in the upcoming retriever and agent notebooks to build powerful GraphRAG applications.

---

**Next:** Continue to [Lab 7 - GraphRAG Retrievers](../Lab_7_Retrievers/) to implement different retrieval patterns.

In [ ]:
# Cleanup
driver.close()
print("Connection closed.")